## Setup

In [1]:
from jax import numpy as jnp, vmap, jit, lax
from jax.tree_util import Partial
from jaxcmr.memorysearch.DataLikelihood import trial_item_count, log_likelihood, predict_and_simulate_pres_and_trial, variable_presentations_likelihood, variable_presentations_data_likelihood, recall_by_item_index
from jaxcmr.memorysearch import InstanceCMR, stop_probability
import numpy as np

In [19]:
from jaxcmr.evaluation import extract_objective_data
from jaxcmr.datasets import load_data, generate_trial_mask, load_parameters

data_path = '../../../../data/{}.h5'
data = load_data(data_path.format('LohnasKahana2014'))
parameters = load_parameters('../../../../data/base_cmr_parameters.json')
parameters['fixed']['mcf_trace_sensitivity'] = 1.0

trial_query = "data['subject'] != -1"
trial_mask = generate_trial_mask(data, trial_query)
# trials, list_lengths, presentations, pres_string_ids, has_repetitions = extract_objective_data(data, trial_mask)

In [20]:
presentations = jnp.array(data['pres_itemnos'])
recall_by_study_position = jnp.array(data['recalls'])
trials = vmap(recall_by_item_index)(presentations, recall_by_study_position)

## By Item Count

In [23]:
item_counts = vmap(trial_item_count)(presentations)
unique_item_counts = np.unique(item_counts)
unique_item_counts

array([20, 34, 40])

In [43]:
total_likelihood = 0

for item_count in unique_item_counts:
    
    likelihood = variable_presentations_likelihood(
        InstanceCMR.create,
        item_count,
        jnp.array(presentations[item_counts==item_count]),
        jnp.array(trials[item_counts==item_count]),
        parameters['fixed'],
    )
    total_likelihood += log_likelihood(likelihood)
    
    print(log_likelihood(likelihood), np.any(likelihood == 0))
    
print(total_likelihood)

34787.277 False
29778.535 False
32994.79 False
97560.6


## Generator

In [50]:
def variable_presentations_data_likelihood(
    model_create_fn,
    presentations,
    trials,
):
    item_counts = vmap(trial_item_count)(presentations)
    functions = [
        Partial(
            variable_presentations_likelihood,
            model_create_fn,
            item_count,
            presentations[item_counts == item_count],
            trials[item_counts == item_count],
        )
        for item_count in np.unique(item_counts)
    ]

    @jit
    def f(parameters):
        log_likelihoods = []
        for fn in functions:
            log_likelihoods.append(fn(parameters))
        return log_likelihood(jnp.vstack(log_likelihoods))

    return f
    
likelihood_fn = variable_presentations_data_likelihood(
    InstanceCMR.create,
    presentations,
    trials,
)

print(likelihood_fn(parameters['fixed']))
%timeit likelihood_fn(parameters['fixed'])

97560.59
116 ms ± 937 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [52]:
def variable_presentations_data_likelihood(
    model_create_fn,
    presentations,
    trials,
):
    item_counts = vmap(trial_item_count)(presentations)
    functions = [
        lambda parameters: variable_presentations_likelihood(
            model_create_fn,
            item_count,
            presentations[item_counts == item_count],
            trials[item_counts == item_count],
            parameters,
        )
        for item_count in np.unique(item_counts)
    ]

    # @jit
    def f(parameters):
        log_likelihoods = []
        for fn in functions:
            log_likelihoods.append(fn(parameters))
        return log_likelihood(jnp.vstack(log_likelihoods))

    return f
    
likelihood_fn = variable_presentations_data_likelihood(
    InstanceCMR.create,
    presentations,
    trials,
)

print(likelihood_fn(parameters['fixed']))
%timeit likelihood_fn(parameters['fixed'])

98984.35
2.23 s ± 105 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [47]:
item_counts = vmap(trial_item_count)(presentations)
model_create_fn = InstanceCMR.create

functions = []
sub_presentations = []
sub_trials = []
sub_item_counts = np.unique(item_counts)
for i in range(len(sub_item_counts)):
    each = sub_item_counts[i]
    sub_presentations.append(jnp.array(presentations[item_counts==each]))
    sub_trials.append(jnp.array(trials[item_counts==each]))
    
    print(sub_presentations[i].shape)
    print(
        log_likelihood(
            variable_presentations_likelihood(
            model_create_fn,
            sub_item_counts[i],
            sub_presentations[i],
            sub_trials[i],
            parameters['fixed'])
        )
    )
    
    
    functions.append(
        lambda parameters: variable_presentations_likelihood(
            model_create_fn,
            sub_item_counts[i],
            sub_presentations[i],
            sub_trials[i],
            parameters,
        )
    )
    
@jit
def likelihood_fn(parameters):
    return log_likelihood(jnp.array([fn(parameters) for fn in functions]))

(840, 40)
34787.277
(420, 40)
29778.535
(420, 40)
32994.79


In [48]:
likelihood_fn(parameters['fixed'])

Array(98984.37, dtype=float32)